# NASA C-MAPSS Predictive Maintenance

A reproducible modernization of the eighth-semester aircraft-engine damage propagation project. The notebook delegates reusable work to the tested `cmapss_maintenance` package and focuses on reasoning, diagnostics, and results.

## Experimental design

- Benchmark: NASA C-MAPSS FD001 (one operating condition, one fault mode).
- Target: piecewise-linear RUL capped at 125 cycles during training.
- Validation: complete engines are held out with `GroupShuffleSplit`.
- Regression selection: lowest NASA asymmetric score on held-out engine endpoints.
- Maintenance policy: probability threshold chosen from an explicit cost matrix.
- Final evaluation: untouched official FD001 test trajectories and RUL truth.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from cmapss_maintenance.config import ExperimentConfig
from cmapss_maintenance.data import add_train_rul, download_dataset, load_fd001
from cmapss_maintenance.modeling import run_experiment
from cmapss_maintenance.reporting import save_artifacts

sns.set_theme(style='whitegrid', context='notebook')
ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data/raw'
OUTPUT_DIR = ROOT / 'artifacts'
CONFIG = ExperimentConfig()

## 1. Acquire and validate the official data

In [ ]:
download_dataset(DATA_DIR)
train_raw, test_raw, test_rul = load_fd001(DATA_DIR)
train = add_train_rul(train_raw, cap=CONFIG.rul_cap)
pd.DataFrame({
    'split': ['train', 'test'],
    'rows': [len(train), len(test_raw)],
    'engines': [train.unit_number.nunique(), test_raw.unit_number.nunique()],
})

## 2. Inspect degradation signals

The plot follows several engines from healthy operation toward failure. Rolling features in the package are causal: each row only uses the current and four preceding cycles.

In [ ]:
sample = train[train.unit_number.isin([1, 25, 50, 75, 100])]
figure, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.lineplot(
    data=sample, x='time_in_cycles', y='sensor_11', hue='unit_number',
    palette='tab10', ax=axes[0],
)
sns.lineplot(
    data=sample, x='time_in_cycles', y='rul_raw', hue='unit_number',
    palette='tab10', legend=False, ax=axes[1],
)
axes[0].set_title('Sensor 11 trajectories')
axes[1].set_title('Remaining useful life')
figure.tight_layout()

## 3. Select models on held-out engines and evaluate the official test set

In [ ]:
result = run_experiment(train, test_raw, test_rul, CONFIG)
validation = pd.DataFrame(result.metrics['validation_regression']).T
validation.sort_values('nasa_score')

In [ ]:
pd.Series(result.metrics['test_regression'], name='official FD001 test').to_frame()

In [ ]:
ordered = result.predictions.sort_values('rul_actual', ascending=False).reset_index(drop=True)
ax = ordered[['rul_actual', 'rul_predicted']].plot(figsize=(13, 5), linewidth=1.8)
ax.set(
    title='Official FD001 test predictions',
    xlabel='Test engines (sorted by actual RUL)',
    ylabel='Cycles',
)
plt.show()

## 4. Cost-aware maintenance decisions

A fixed 0.5 cutoff is not automatically optimal when missed maintenance and unnecessary maintenance have different costs. The threshold below is selected using held-out engines only. Values are illustrative and must be replaced by operator-specific economics and safety constraints before real use.

In [ ]:
pd.Series({
    'selected_probability_threshold': result.threshold,
    **result.metrics['test_maintenance_value'],
}).to_frame('value')

In [ ]:
maintenance_cases = result.predictions.query(
    'maintenance_actual == 1 or maintenance_predicted == 1'
)
maintenance_cases.sort_values('maintenance_probability', ascending=False)

## 5. Save reproducible artifacts

In [ ]:
save_artifacts(result, OUTPUT_DIR)
print(json.dumps(result.metrics, indent=2))
sorted(path.name for path in OUTPUT_DIR.iterdir())

## Limitations and next steps

FD001 is simulated and represents one condition and one fault mode. A production PHM system would require uncertainty intervals, temporal and fleet drift monitoring, calibrated probabilities, maintenance-capacity constraints, human review, safety assurance, and validation on representative operational data. FD002-FD004 provide useful stress tests for multiple operating conditions and fault modes.